# Демо 3. RAG на LangChain і оцінювання

У демо 2 ми написали RAG-пайплайн вручну. Тут збираємо той самий пайплайн із готових компонентів
[LangChain](https://python.langchain.com/), додаємо два способи покращити пошук — **гібридний пошук**
і **переранжування** — і, головне, **вимірюємо** якість системи на наборі питань із відомими відповідями.

Ретрив і генерацію оцінюємо окремо:
- **ретрив** — чи потрапляє фрагмент із потрібного документа до знайдених (hit rate@k, MRR);
- **генерація** — чи правильна відповідь, чи посилається вона на правильне джерело
  і чи відмовляється система відповідати, коли відповіді в документах немає.

_Для цього ноутбука потрібен GPU ._

In [4]:
from time import perf_counter as _demo_clock
_demo_start = _demo_clock()
print("Початок заміру часу")

Початок заміру часу


In [5]:
# LangChain розбитий на пакети: core — базові абстракції, text-splitters — розбиття тексту,
# community — інтеграції (FAISS, BM25), huggingface — моделі з Hugging Face
!pip install -q "sentence-transformers>=5" pypdf langchain-core langchain-text-splitters langchain-community langchain-huggingface faiss-cpu rank_bm25

In [6]:
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader
from sentence_transformers import CrossEncoder
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"Пристрій: {device}")

Пристрій: cuda


## 1. Документи

Ті самі 14 документів, що в демо 1 і 2. LangChain має готові завантажувачі PDF, але колонтитули
й скани все одно доводиться обробляти самостійно, тому текст витягуємо так само, як раніше,
а потім загортаємо кожен документ в обʼєкт `Document`: текст плюс метадані.

In [7]:
# @title Завантаження й очищення документів (як у демо 1)
BASE_URL = "https://docs.ztu.edu.ua/"
DOCUMENTS = [
    ("275", "Положення про оцінювання результатів навчання (чинна версія 4.0)", {"mdocs-file": 275}),
    ("275-v3.1", "Положення про оцінювання результатів навчання (стара версія 3.1)",
     {"mdocs-version": "polozhennja pro ocinjuvannja znan yekts_nove.pdf-v3.1", "mdocs-file": 275, "mdocs-url": "false"}),
    ("277", "Положення про порядок та умови обрання здобувачами вищої освіти навчальних дисциплін", {"mdocs-file": 277}),
    ("291", "Положення про проведення практики здобувачів вищої освіти", {"mdocs-file": 291}),
    ("340", "Правила призначення академічних стипендій", {"mdocs-file": 340}),
    ("344", "Положення про стипендію ректора", {"mdocs-file": 344}),
    ("348", "Положення про користування гуртожитками", {"mdocs-file": 348}),
    ("1219", "Правила внутрішнього розпорядку в студентських гуртожитках", {"mdocs-file": 1219}),
    ("1199", "Кодекс академічної доброчесності", {"mdocs-file": 1199}),
    ("4389", "Положення про студентського омбудсмана", {"mdocs-file": 4389}),
    ("336", "Положення про старосту академічної групи", {"mdocs-file": 336}),
    ("3320", "ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року", {"mdocs-file": 3320}),
    ("3329", "ОПП 121 «Інженерія програмного забезпечення» (бакалавр), редакція 2024 року", {"mdocs-file": 3329}),
    ("283", "Положення про переведення здобувачів вищої освіти", {"mdocs-file": 283}),
]


PDF_DIR = Path("ztu_pdfs")
PDF_DIR.mkdir(exist_ok=True)


def fetch(params, attempts=3):
    # Сайт іноді обриває зʼєднання, тому пробуємо кілька разів
    for attempt in range(attempts):
        try:
            response = requests.get(BASE_URL, params=params, timeout=120)
            response.raise_for_status()
            return response.content
        except requests.RequestException:
            if attempt == attempts - 1:
                raise
            time.sleep(5)


def download_pdf(doc_id, params):
    # Зберігаємо PDF на диск, щоб під час повторного запуску не завантажувати їх знову
    path = PDF_DIR / f"{doc_id}.pdf"
    if not path.exists():
        path.write_bytes(fetch(params))
    return PdfReader(path)


HEADER_LINE = re.compile(
    r"^(Житомирська|політехніка|Міністерство освіти.*|Державний університет «Житомирська політехніка»|"
    r"Система управління якістю.*|П[\s\-–]*[CС]?[АA]?[ГG]?[\s\d.\-–]*|[\d.\-–\s/_]+|Випуск.*)$",
    re.IGNORECASE,
)
HEADER_LINES = 12


def page_text(page):
    lines = (page.extract_text() or "").split("\n")
    head = [line for line in lines[:HEADER_LINES] if not HEADER_LINE.match(line.strip())]
    return " ".join(head + lines[HEADER_LINES:])


MIN_CHARS_PER_PAGE = 100


def document_text(reader):
    return re.sub(r"\s+", " ", " ".join(page_text(page) for page in reader.pages)).strip()


def is_scan(text, reader):
    return len(text) < MIN_CHARS_PER_PAGE * len(reader.pages)


documents = {}
for doc_id, title, params in DOCUMENTS:
    reader = download_pdf(doc_id, params)
    text = document_text(reader)
    if is_scan(text, reader):
        print(f"{doc_id}: текстового шару немає, потрібне OCR — пропущено")
        continue
    documents[doc_id] = {"title": title, "text": text}
    print(f"{doc_id}: {len(reader.pages)} сторінок, {len(text)} символів — {title}")

275: 23 сторінок, 24944 символів — Положення про оцінювання результатів навчання (чинна версія 4.0)
275-v3.1: 13 сторінок, 10505 символів — Положення про оцінювання результатів навчання (стара версія 3.1)
277: 16 сторінок, 15945 символів — Положення про порядок та умови обрання здобувачами вищої освіти навчальних дисциплін
291: 17 сторінок, 14603 символів — Положення про проведення практики здобувачів вищої освіти
340: 30 сторінок, 35706 символів — Правила призначення академічних стипендій
344: 10 сторінок, 2455 символів — Положення про стипендію ректора
348: 22 сторінок, 25608 символів — Положення про користування гуртожитками
1219: 12 сторінок, 9159 символів — Правила внутрішнього розпорядку в студентських гуртожитках
1199: 37 сторінок, 56918 символів — Кодекс академічної доброчесності
4389: 18 сторінок, 20685 символів — Положення про студентського омбудсмана
336: 8 сторінок, 3560 символів — Положення про старосту академічної групи
3320: 16 сторінок, 31670 символів — ОПП F2 «Інженері

In [8]:
docs = [
    Document(page_content=doc["text"], metadata={"doc_id": doc_id, "title": doc["title"]})
    for doc_id, doc in documents.items()
]
print(docs[0].metadata, docs[0].page_content[:200])

{'doc_id': '275', 'title': 'Положення про оцінювання результатів навчання (чинна версія 4.0)'} ЗАТВЕРДЖЕНО Наказ Державного університету «Житомирська політехніка» 15 вересня 2025 р. № 327/од Ректор Віктор ЄВДОКИМОВ ПОЛОЖЕННЯ про оцінювання результатів навчання здобувачів вищої освіти у Державно


## 2. Розбиття на фрагменти

`RecursiveCharacterTextSplitter` пробує різати текст за першим роздільником зі списку; якщо шматок
усе ще задовгий, переходить до наступного роздільника. Наш список: кінець речення, крапка з комою,
пробіл і, в крайньому разі, будь-який символ. Метадані документа копіюються в кожен фрагмент.
Як і раніше, дописуємо на початок фрагмента назву документа.

In [9]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, separators=[". ", "; ", " ", ""], keep_separator="end"
)
chunks = splitter.split_documents(docs)
for chunk in chunks:
    chunk.page_content = f"{chunk.metadata['title']}. {chunk.page_content}"

print(f"Фрагментів: {len(chunks)}")
print(chunks[100].metadata)
print(chunks[100].page_content)

Фрагментів: 385
{'doc_id': '340', 'title': 'Правила призначення академічних стипендій'}
Правила призначення академічних стипендій. 1.9. Студенти, аспіранти (докторанти), які реалізують право на академічну мобільність і зберігають статус здобувача вищої освіти (наукового ступеня) за денною формою навчання (навчаються з відривом від виробництва) у Житомирській політехніці, протягом строку навчання за програмою академічної мобільності в іншому навчальному закладі поза межами України отримують стипендію, призначену відповідно до цих Правил, у разі, коли умовами договору про навчання за програмою академічної мобільності, укладеного ними з Житомирською політехнікою, регулярна безповоротна фіксована фінансова підтримка у грошовій формі протягом всього строку навчання за програмою академічної мобільності в іншому навчальному закладі не передбачена або передбачена у розмірі, що у перерахунку на національну грошову одиницю на дату укладення договору про академічну мобільність є меншою за розмір 

## 3. Векторне сховище

`HuggingFaceEmbeddings` — обгортка LangChain над sentence-transformers. Для документів і запитів
задаємо різні параметри: запит кодується з промптом `query`, як у демо 1.
[FAISS](https://github.com/facebookresearch/faiss) — бібліотека векторного пошуку від Meta;
для нормалізованих векторів обираємо скалярний добуток як міру подібності.

In [10]:
EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": device, "model_kwargs": {"dtype": dtype}},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 16},
    query_encode_kwargs={"normalize_embeddings": True, "prompt_name": "query"},
)
vector_store = FAISS.from_documents(chunks, embeddings, distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT)

for doc, score in vector_store.similarity_search_with_score("Коли обирають старосту на першому курсі?", k=3):
    print(f"{score:.2f}  [{doc.metadata['doc_id']}] {doc.page_content[:120]}…")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

... (4 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

0.60  [336] Положення про старосту академічної групи. У своїй роботі староста академічної групи співпрацює з куратором та ментором г…
0.53  [336] Положення про старосту академічної групи. До виборів обов’язки старости групи може виконувати один із студентів групи за…
0.50  [4389] Положення про студентського омбудсмана. 6.4. Студентський омбудсман взаємодіє з Молодіжною радою при Житомирській обласн…


## 4. Лексичний пошук BM25

BM25 — класичний алгоритм пошуку за збігом слів: фрагмент отримує тим більшу оцінку, чим більше в ньому
слів запиту і чим рідше ці слова трапляються в інших фрагментах. BM25 не розуміє синонімів і,
без додаткової обробки, різних форм того самого слова («гуртожиток» і «гуртожитку» для нього різні слова),
зате точно знаходить рідкісні слова, номери й коди.

In [11]:
CANDIDATES = 20


def tokenize(text):
    return re.findall(r"\w+", text.lower())


bm25 = BM25Retriever.from_documents(chunks, k=CANDIDATES, preprocess_func=tokenize)


def vector_search(query, k):
    return vector_store.similarity_search(query, k=k)


def bm25_search(query, k):
    return bm25.invoke(query)[:k]


def show(search, query, k=3):
    print(f"{search.__name__}: {query}")
    for doc in search(query, k):
        print(f"  [{doc.metadata['doc_id']}] {doc.page_content[:110]}…")
    print()


for query in ["До якої дати потрібно оплатити гуртожиток за перший семестр?",
              "Яку допомогу отримує студент в академічній відпустці за медичними показаннями?"]:
    show(vector_search, query)
    show(bm25_search, query)

vector_search: До якої дати потрібно оплатити гуртожиток за перший семестр?
  [348] Положення про користування гуртожитками. 5.2. Студент вносить авансовий платіж оплати за 1 семестр (5 місяців)…
  [348] Положення про користування гуртожитками. Мешканці гуртожитків Житомирської політехніки зобов’язані: - знати й …
  [348] Положення про користування гуртожитками. Це є єдиною підставою для поселення і проживання студента у вказаному…

bm25_search: До якої дати потрібно оплатити гуртожиток за перший семестр?
  [348] Положення про користування гуртожитками. 3.9. Здобувачам освіти, які поселяються до гуртожитку Житомирської по…
  [340] Правила призначення академічних стипендій. Цей показник встановлюється у відсотках (у діапазоні від 40 до 45) …
  [340] Правила призначення академічних стипендій. Рішенням ректора Житомирської політехніки таким особам може встанов…

vector_search: Яку допомогу отримує студент в академічній відпустці за медичними показаннями?
  [340] Правила призначення академ

Фрагмент з відповіддю на перше питання містить слова «авансовий платіж оплати… до 1 вересня»,
а не «оплатити»: семантичний пошук знаходить його, а BM25 — лише серед нижчих результатів.
У другому питанні рідкісний вираз «за медичними показаннями» дослівно є у фрагменті з відповіддю,
і BM25 ставить цей фрагмент першим, а семантичний пошук — нижче.

## 5. Гібридний пошук

Два списки результатів обʼєднуємо методом **reciprocal rank fusion** (RRF): фрагмент на позиції $r$
у кожному списку отримує $\frac{1}{60 + r}$ балів, бали з обох списків додаються.
RRF використовує лише позиції, тому не потребує зведення оцінок BM25 і косинусної подібності до однієї шкали.

In [12]:
def rrf(rankings, c=60):
    scores, by_content = {}, {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            scores[doc.page_content] = scores.get(doc.page_content, 0) + 1 / (c + rank)
            by_content[doc.page_content] = doc
    return [by_content[content] for content in sorted(scores, key=scores.get, reverse=True)]


def hybrid_search(query, k):
    return rrf([vector_search(query, CANDIDATES), bm25_search(query, CANDIDATES)])[:k]


show(hybrid_search, "Якому діапазону балів відповідає оцінка FX?")

hybrid_search: Якому діапазону балів відповідає оцінка FX?
  [3320] ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року. Оцінювання Оцінювання навчальних д…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Наявні помилки (неточності та ін.) 35 – 59 б…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Під час поточного контролю оцінюванню можуть…



## 6. Переранжування

Модель ембедингів — **bi-encoder**: запит і фрагмент кодуються окремо, тому ембединги фрагментів можна порахувати
заздалегідь. **Cross-encoder** отримує запит і фрагмент разом і оцінює їхню відповідність точніше,
але для кожної пари потрібен окремий прохід моделі. Тому cross-encoder застосовують лише до кількох десятків
кандидатів, знайдених швидким пошуком.
Використовуємо багатомовну модель [bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3).

In [13]:
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device=device, model_kwargs={"dtype": dtype})


def rerank_search(query, k):
    candidates = hybrid_search(query, CANDIDATES)
    scores = reranker.predict([(query, doc.page_content) for doc in candidates])
    return [candidates[i] for i in np.argsort(-scores)[:k]]


show(rerank_search, "Якому діапазону балів відповідає оцінка FX?")

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

rerank_search: Якому діапазону балів відповідає оцінка FX?
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Наявні помилки (неточності та ін.) 35 – 59 б…
  [3320] ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року. Оцінювання Оцінювання навчальних д…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Значення вагового коефіцієнта для кожного ви…



## 7. Бенчмарк

Набір питань складено **до** налаштування системи. Для кожного питання вказано документи, у яких є відповідь,
регулярний вираз, за яким можна впізнати фрагмент із відповіддю, та еталонну відповідь. Три питання не мають відповіді в документах: на них система має відмовитися відповідати.

In [14]:
BENCHMARK = [
    ("До якої дати потрібно оплатити гуртожиток за перший семестр?", ["348"], r"1 вересня", "До 1 вересня поточного року."),
    ("До якої дати оплачують гуртожиток за другий семестр?", ["348"], r"1 лютого", "До 1 лютого поточного року."),
    ("З якої до якої години в гуртожитку має бути тиша?", ["1219"], r"23:00 години і до 7:00", "З 23:00 до 7:00."),
    ("У який час гості можуть перебувати в гуртожитку?", ["1219", "348"], r"0?8:00 до 22:00", "З 8:00 до 22:00."),
    ("Чи можна тримати в гуртожитку домашню тварину?", ["1219", "348"], r"утримувати тварин", "Ні, утримувати тварин у гуртожитку заборонено."),
    ("На який строк укладають договір найму житла в гуртожитку?", ["348"], r"терміном на один навчальний рік", "На один навчальний рік."),
    ("За який час потрібно звільнити гуртожиток після відрахування?", ["1219"], r"протягом двох тижнів", "Протягом двох тижнів від дня виходу наказу."),
    ("Коли обирають старосту на першому курсі?", ["336"], r"після двох тижнів", "На початку навчального року, після двох тижнів навчального процесу."),
    ("На який строк обирається староста групи?", ["336"], r"на весь період навчання", "На весь період навчання."),
    ("Скільки балів потрібно для оцінки «добре» B?", ["275"], r"82 ?– ?89", "82–89 балів."),
    ("Якому діапазону балів відповідає оцінка FX?", ["275"], r"35 ?– ?59", "35–59 балів."),
    ("Яку частку кредитів становлять вибіркові дисципліни?", ["277"], r"25 відсотків",
     "Не менше 25 відсотків загальної кількості кредитів ЄКТС (не менше 10 відсотків для спеціальностей з додатковим регулюванням)."),
    ("До якої дати викладачі вносять анотації вибіркових дисциплін?", ["277"], r"30 квітня", "До 30 квітня."),
    ("Як часто виплачують стипендію?", ["340"], r"один раз на місяць", "Один раз на місяць."),
    ("За яким показником формують рейтинг для призначення академічної стипендії?", ["340"], r"середнім балом", "За середнім балом семестрового контролю."),
    ("Яку допомогу отримує студент-бюджетник в академічній відпустці за медичними показаннями?", ["340"], r"50 відсотків мінімальної",
     "Щомісячну допомогу в розмірі 50 відсотків мінімальної академічної стипендії."),
    ("Для кого встановлюється стипендія ректора?", ["344"], r"встановлюється для студентів денної",
     "Для студентів денної форми навчання із суттєвими досягненнями в навчанні, науковій, творчій і громадській діяльності."),
    ("На який строк обирають студентського омбудсмана?", ["4389"], r"терміном на 1 рік", "На 1 рік."),
    ("До якої дати проводять вибори студентського омбудсмана?", ["4389"], r"до 01 жовтня", "До 1 жовтня кожного навчального року."),
    ("Що таке самоплагіат?", ["1199"], r"Самоплагіат –", "Оприлюднення власних раніше опублікованих результатів без зазначення джерела попереднього оприлюднення."),
    ("Що таке фабрикація в академічній діяльності?", ["1199"], r"Фабрикація –", "Вигадування даних чи фактів про результати власної академічної діяльності."),
    ("Який обсяг бакалаврської програми «Інженерія програмного забезпечення» в кредитах ЄКТС?", ["3320", "3329"], r"240 кредит", "240 кредитів ЄКТС."),
    ("Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?", ["3320"], r"спеціальн\w* F2", "F2."),
    ("Скільки коштує обід у студентській їдальні?", [], None, None),
    ("Який графік роботи бібліотеки університету?", [], None, None),
    ("Скільки коштує рік навчання на контракті за спеціальністю «Інженерія програмного забезпечення»?", [], None, None),
]
answerable = [(q, gold, evidence) for q, gold, evidence, _ in BENCHMARK if gold]
print(f"Питань: {len(BENCHMARK)}, з них без відповіді в документах: {len(BENCHMARK) - len(answerable)}")

Питань: 26, з них без відповіді в документах: 3


### Метрики ретриву

Фрагмент вважаємо правильним, якщо він походить із правильного документа і містить вираз-доказ.
Перевіряти лише документ замало: у довгому документі десятки фрагментів, і більшість із них відповіді не містять.

- **hit rate@k** — частка питань, для яких правильний фрагмент є серед `k` знайдених;
- **MRR** (mean reciprocal rank) — середнє за питаннями значення $\frac{1}{r}$, де $r$ — позиція першого
  правильного фрагмента; якщо його немає серед перших 10, замість $\frac{1}{r}$ береться 0. MRR враховує не лише те, чи знайдено фрагмент, а й те, наскільки високо.

Питання без відповіді в ретриві не оцінюємо: правильного фрагмента для них немає.

In [15]:
def first_hit_rank(results, gold, evidence):
    for rank, doc in enumerate(results, start=1):
        if doc.metadata["doc_id"] in gold and re.search(evidence, doc.page_content):
            return rank
    return None


def hit_rate(ranks, k):
    return np.mean([rank is not None and rank <= k for rank in ranks])


def mrr(ranks):
    return np.mean([1 / rank if rank else 0 for rank in ranks])


def retrieval_metrics(search):
    ranks = [first_hit_rank(search(question, 10), gold, evidence) for question, gold, evidence in answerable]
    return {"метод": search.__name__, "hit@1": hit_rate(ranks, 1), "hit@5": hit_rate(ranks, 5), "MRR": mrr(ranks)}


base_retrieval_metrics = pd.DataFrame([retrieval_metrics(search) for search in [vector_search, bm25_search, hybrid_search, rerank_search]])
display(base_retrieval_metrics.round(2))

,метод,hit@1,hit@5,MRR
0,vector_search,0.61,0.87,0.74
1,bm25_search,0.52,0.65,0.59
2,hybrid_search,0.61,0.91,0.76
3,rerank_search,0.83,0.96,0.90


На українських текстах BM25 сам по собі зазвичай поступається семантичному пошуку: слова мають багато форм,
а питання рідко повторюють формулювання документа. Переранжування найсильніше впливає на hit@1:
cross-encoder бачить запит і фрагмент разом і частіше ставить правильний фрагмент на перше місце.
Висновки варто робити обережно: на 23 питаннях одне питання — це понад 4 відсоткові пункти.

## 8. RAG-ланцюжок на LangChain

Генеративна модель та сама, що в демо 2. Загортаємо її в `HuggingFacePipeline`, а `ChatHuggingFace`
застосовує до повідомлень шаблон чату моделі.

In [16]:
GENERATOR_MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL_NAME)
llm = AutoModelForCausalLM.from_pretrained(GENERATOR_MODEL_NAME, dtype=dtype).to(device)

generator = pipeline(
    "text-generation", model=llm, tokenizer=tokenizer,
    max_new_tokens=300, do_sample=False, temperature=None, top_p=None, top_k=None, return_full_text=False,
)
chat = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=generator), tokenizer=tokenizer)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Ланцюжок записуємо мовою LCEL (LangChain Expression Language): оператор `|` передає вихід одного кроку
на вхід наступного. Ланцюжок повертає і відповідь, і знайдені фрагменти: вони потрібні для перевірки цитувань.

In [17]:
SYSTEM_PROMPT = (
    "Ти — асистент, що відповідає на питання студентів Житомирської політехніки. "
    "Відповідай лише на основі наданих фрагментів документів. "
    "Після кожного твердження вказуй у квадратних дужках номер фрагмента, на якому воно ґрунтується, наприклад [2]. "
    "Якщо у фрагментах немає відповіді на питання, відповідай лише: «У наданих документах немає відповіді.» "
    "Відповідай українською мовою, коротко."
)
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Фрагменти документів:\n\n{context}\n\nПитання: {question}"),
])


def format_docs(docs):
    return "\n\n".join(f"[{n}] {doc.page_content}" for n, doc in enumerate(docs, start=1))


retriever = RunnableLambda(lambda question: rerank_search(question, 5))
answer_chain = (
    RunnableLambda(lambda x: {"context": format_docs(x["docs"]), "question": x["question"]})
    | prompt
    | chat
    | StrOutputParser()
)
rag_chain = {"docs": retriever, "question": RunnablePassthrough()} | RunnablePassthrough.assign(answer=answer_chain)

result = rag_chain.invoke("Чи можна тримати в гуртожитку домашню тварину?")
print(result["answer"])
for n, doc in enumerate(result["docs"], start=1):
    print(f"  [{n}] {doc.metadata['doc_id']}: {doc.metadata['title']}")

Ні, в гуртожитку домашню тварину тримати заборонено [1].
  [1] 348: Положення про користування гуртожитками
  [2] 348: Положення про користування гуртожитками
  [3] 348: Положення про користування гуртожитками
  [4] 1219: Правила внутрішнього розпорядку в студентських гуртожитках
  [5] 1219: Правила внутрішнього розпорядку в студентських гуртожитках


## 9. Оцінювання відповідей

Для кожного питання бенчмарку перевіряємо:
- **правильність**: для питань із відповіддю — чи збігається відповідь з еталонною; для питань без відповіді — чи відмовилася система;
- **цитування**: чи є серед фрагментів, на які посилається відповідь, фрагмент із правильного документа.

Правильність вільної відповіді важко перевірити правилом: «до 1 вересня» і «не пізніше першого вересня» — одна відповідь.
Тому використовуємо підхід **LLM як суддя**: інша (тут — та сама) модель порівнює відповідь з еталоном.
Суддя теж помиляється, тому його вердикти варто вибірково перевіряти вручну.

In [18]:
REFUSAL = "немає відповіді"
JUDGE_PROMPT = (
    "Порівняй відповідь системи з еталонною відповіддю на питання.\n\n"
    "Питання: {question}\nЕталонна відповідь: {reference}\nВідповідь системи: {answer}\n\n"
    "Чи містить відповідь системи ту саму інформацію, що й еталонна, без суперечностей? "
    "Відповідай одним словом: «так» або «ні»."
)


def judge(question, reference, answer):
    verdict = chat.invoke(JUDGE_PROMPT.format(question=question, reference=reference, answer=answer)).content
    return verdict.strip().lower().startswith("так")


def cited_doc_ids(answer, retrieved):
    # Номери на зразок [2] у відповіді перетворюємо на ідентифікатори документів
    numbers = [int(n) for n in re.findall(r"\[(\d+)\]", answer)]
    return [retrieved[n - 1] for n in numbers if 0 < n <= len(retrieved)]


evaluation_contexts = {}


def evaluate(question, gold, reference):
    result = rag_chain.invoke(question)
    evaluation_contexts[question] = result["docs"]
    answer = result["answer"]
    retrieved = [doc.metadata["doc_id"] for doc in result["docs"]]
    if gold:
        correct = judge(question, reference, answer)
        citation_ok = any(doc_id in gold for doc_id in cited_doc_ids(answer, retrieved))
    else:
        correct = REFUSAL in answer.lower()
        citation_ok = None
    return {"питання": question, "правильно": correct, "цитування": citation_ok,
            "документи": gold, "знайдено": retrieved, "відповідь": answer}


evaluation = pd.DataFrame([evaluate(question, gold, reference) for question, gold, _, reference in BENCHMARK])
has_answer = evaluation["цитування"].notna()
print(f"Правильних відповідей на питання з відповіддю: {evaluation[has_answer]['правильно'].mean():.0%}")
print(f"Коректних цитувань: {evaluation[has_answer]['цитування'].astype(bool).mean():.0%}")
print(f"Правильних відмов на питання без відповіді: {evaluation[~has_answer]['правильно'].mean():.0%}")

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Правильних відповідей на питання з відповіддю: 91%
Коректних цитувань: 96%
Правильних відмов на питання без відповіді: 100%


## 10. Аналіз відмов

Найкорисніша частина оцінювання — не підсумкове число, а розбір помилок. Для кожної неправильної відповіді
дивимося, чи був серед знайдених фрагментів правильний документ:
- якщо **не був** — це промах ретриву, і виправляти треба пошук (розбиття, модель ембедингів, гібридний пошук);
- якщо **був** — модель не скористалася контекстом або скористалася неправильно, і виправляти треба промпт чи модель.

In [19]:
def retrieval_verdict(row):
    if not row["документи"] or any(doc_id in row["документи"] for doc_id in row["знайдено"]):
        return "ретрив знайшов"
    return "промах ретриву"


def describe_failure(row):
    print(f"Питання: {row['питання']}")
    print(f"  правильний документ: {row['документи']}, знайдено: {row['знайдено']} — {retrieval_verdict(row)}")
    print(f"  правильно: {row['правильно']}, цитування: {row['цитування']}")
    print(f"  відповідь: {row['відповідь']}\n")


failures = evaluation[(~evaluation["правильно"]) | (evaluation["цитування"] == False)]
for _, row in failures.iterrows():
    describe_failure(row)

Питання: Для кого встановлюється стипендія ректора?
  правильний документ: ['344'], знайдено: ['344', '340', '344', '340', '340'] — ретрив знайшов
  правильно: False, цитування: True
  відповідь: Для студентів денної форми навчання, які мають суттєві досягнення у навчанні, науковій, творчій та громадській діяльності, є переможцями міжнародних та всеукраїнських олімпіад, конкурсів та змагань [1].

Питання: Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?
  правильний документ: ['3320'], знайдено: ['3329', '3329', '3320', '3329', '3329'] — ретрив знайшов
  правильно: False, цитування: False
  відповідь: Шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі — 121 [2].



Кожну неправильну відповідь відносять до однієї з двох груп:
- **помилка оцінювання**: відповідь правильна, але суддя помилився (відповідь сформульована інакше або містить більше деталей, ніж еталон)
  або еталон неоднозначний (відповідь спирається на інше місце документа, ніж еталон, і не суперечить документу);
- **помилка системи**: промах ретриву, галюцинація попри наявний контекст або суперечливі джерела —
  як у питанні про шифр спеціальності, де чинна редакція програми сама містить старий шифр 121 (див. демо 2).

Перед тим як змінювати систему, варто перевірити, чи правильно оцінено відповідь.

## Підсумок

- LangChain дає готові компоненти — розбивачі тексту, обгортки моделей, векторні сховища, ретривери — і спосіб
  зʼєднати їх у ланцюжок; логіку пошуку, як-от гібридний пошук чи переранжування, легко додати власними функціями.
- BM25 і семантичний пошук помиляються на різних питаннях, тому їхнє поєднання зазвичай краще за кожен окремо.
- Cross-encoder точніше оцінює відповідність фрагмента запиту, але працює лише на невеликій кількості кандидатів.
- Ретрив і генерацію оцінюють окремо: так видно, де саме виникає помилка.
- Для автоматичного оцінювання відповідей існують готові бібліотеки, наприклад [RAGAS](https://docs.ragas.io/);
  вони теж спираються на LLM як суддю.

In [20]:
_demo_main_seconds = _demo_clock() - _demo_start
print(f"Час виконання основного демо: {_demo_main_seconds:.2f} с ({_demo_main_seconds / 60:.2f} хв)")

Час виконання основного демо: 371.59 с (6.19 хв)


## 0.1. Відповіді на питання 21–30

Відповіді за результатами виконання демо та додаткових експериментів у Kaggle.

### 21. Як довго виконувалися клітинки ноутбуку 03_rag_langchain_eval ? Чи були проблеми з запуском, які?

Основне демо — 371,59 с (6,19 хв), додаткові завдання — 10,38 с, загалом — 381,97 с (6,37 хв). Пристрій — CUDA. Помилок виконання не було. Hugging Face попередив про неавторизовані запити, а Transformers — про послідовне використання pipeline на GPU. Обидва попередження не завадили завершенню.

### 22. Чим RecursiveCharacterTextSplitter (розділ 2) відрізняється від розбиття за реченнями з демо 2? Для чого задано параметр keep_separator="end" ?

У демо 2 текст спочатку ділиться на речення регулярним виразом, а потім речення збираються у фрагменти. `RecursiveCharacterTextSplitter` рекурсивно переходить до дрібніших роздільників у заданому порядку: `. `, `; `, пробіл, порожній рядок (посимвольне розбиття). Розмір фрагмента — 1000 символів, перекриття — 200. `keep_separator="end"` залишає роздільник у кінці попередньої частини, наприклад крапку разом із реченням.

### 23. Чому у розділі 3 для FAISS обрано DistanceStrategy.MAX_INNER_PRODUCT ? Чи можна було б так зробити, якби ембединги не були нормалізовані?

Для нормалізованих ембедингів скалярний добуток дорівнює косинусній подібності. Тому `MAX_INNER_PRODUCT` дозволяє FAISS шукати найближчі за напрямом вектори без додаткового ділення. Без нормалізації скалярний добуток теж можна застосувати, але він уже залежатиме від довжини векторів і може давати інший рейтинг, ніж косинус.

### 24. Знайдіть власне питання до документів, на якому BM25 знаходить правильний фрагмент вище, ніж семантичний пошук, і питання, на якому навпаки (розділ 4). Поясніть, чому так відбувається.

Перевіряю власні формулювання з точними термінами та перефразуваннями. Позицію правильного фрагмента визначаю за документом і виразом-доказом, як у розділі 7. Якщо доказу в Top-20 немає, записую промах.

In [21]:
own_retrieval_questions = [
    ("Що означає термін «Самоплагіат» у Кодексі академічної доброчесності?", ["1199"], r"Самоплагіат –"),
    ("Фабрикація: що вважають порушенням академічної доброчесності?", ["1199"], r"Фабрикація –"),
    ("Скільки балів відповідає оцінці FX за ЄКТС?", ["275"], r"35 ?– ?59"),
    ("Як довго студент може залишатися в кімнаті після того, як його виключили з університету?", ["1219"], r"протягом двох тижнів"),
    ("Як часто студент отримує гроші від академічної стипендії?", ["340"], r"один раз на місяць"),
    ("Коли треба внести гроші за проживання у другому півріччі навчального року?", ["348"], r"1 лютого"),
    ("На скільки часу призначають представника академічної групи?", ["336"], r"на весь період навчання"),
]
rank_rows = []
for question, gold, evidence in own_retrieval_questions:
    v = first_hit_rank(vector_search(question, 20), gold, evidence)
    b = first_hit_rank(bm25_search(question, 20), gold, evidence)
    winner = "BM25" if (b or 21) < (v or 21) else "Semantic" if (v or 21) < (b or 21) else "Однаково"
    rank_rows.append({"питання": question, "BM25": b, "Semantic": v, "кращий метод": winner})
rank_table = pd.DataFrame(rank_rows)
display(rank_table)

,питання,BM25,Semantic,кращий метод
0,Що означає термін «Самоплагіат» у Кодексі акад...,7.0,1,Semantic
1,Фабрикація: що вважають порушенням академічної...,7.0,20,BM25
2,Скільки балів відповідає оцінці FX за ЄКТС?,1.0,1,Однаково
3,Як довго студент може залишатися в кімнаті піс...,NaN,2,Semantic
4,Як часто студент отримує гроші від академічної...,5.0,2,Semantic
5,Коли треба внести гроші за проживання у другом...,12.0,1,Semantic
6,На скільки часу призначають представника акаде...,1.0,1,Однаково


| Питання | BM25 | Semantic |
|---|---:|---:|
| Фабрикація: що вважають порушенням академічної доброчесності? | 7 | 20 |
| Коли треба внести гроші за проживання у другому півріччі навчального року? | 12 | 1 |

У першому випадку BM25 виграв завдяки точному терміну «фабрикація». У другому семантичний пошук краще зіставив перефразування з формулюванням про оплату другого семестру. Менша позиція означає кращий результат.

### 25. Обчисліть "вручну" (за допомогою Python коду без використання допоміжних діб) бал RRF для фрагмента, що стоїть на 2-й позиції у векторному пошуку й на 5-й позиції в BM25 (розділ 5). Чому RRF використовує лише позиції, а не самі оцінки методів пошуку?

При константі $c=60$ бал становить $1/(60+2)+1/(60+5)\approx0.03151365$. RRF використовує позиції, бо BM25 та косинусна подібність мають різні шкали; їхні сирі оцінки не можна безпосередньо додавати без узгодження.

In [22]:
rrf_score = 1 / (60 + 2) + 1 / (60 + 5)
print(f"RRF: {rrf_score:.8f}")

RRF: 0.03151365


### 26. Чому cross-encoder (розділ 6) застосовують лише до 20 кандидатів, а не до всіх фрагментів корпусу? Скільки проходів моделі знадобилося б на одне питання, щоб переранжувати весь корпус?

Cross-encoder обробляє запит і фрагмент разом, тому для кожної пари потрібне окреме оцінювання. На відміну від ембедингів документів, цю оцінку не можна обчислити заздалегідь для всіх можливих запитів. Для цілого корпусу потрібно оцінити `len(chunks)` пар, а для Top-20 — не більше 20. Пакетна обробка може виконати кілька пар за один батч.

У корпусі 385 фрагментів, тобто потрібно оцінити 385 пар замість не більше ніж 20. Це приблизно у 19,25 раза більше пар (не обов'язково стільки ж окремих батчів).

In [23]:
print(f"Фрагментів у корпусі: {len(chunks)}")
print(f"Оцінювань пар для всього корпусу: {len(chunks)}")
print(f"Оцінювань пар для reranking: не більше {CANDIDATES}")

Фрагментів у корпусі: 385
Оцінювань пар для всього корпусу: 385
Оцінювань пар для reranking: не більше 20


### 27. Додайте до бенчмарку (розділ 7) три власні питання: одне з відповіддю, одне, що перефразовує наявне питання, і одне без відповіді в документах. Як змінилися метрики ретриву?

Додаю питання про виняток щодо вибіркових дисциплін, перефразування питання про старосту та питання про пароль Wi-Fi, якого не очікую в корпусі. Початковий бенчмарк не змінюю. Метрики перераховую для розширеного набору; питання без відповіді, як і раніше, не входить у знаменник ретриву. Еталон першого питання взято з відповіді про вибіркові дисципліни в початковому бенчмарку.

In [24]:
extra_benchmark = [
    ("Яка мінімальна частка вибіркових дисциплін для спеціальностей з додатковим регулюванням?", ["277"], r"10\s+відсотків", "Не менше 10 відсотків загальної кількості кредитів ЄКТС."),
    ("На якому етапі навчання першокурсники обирають старосту своєї групи?", ["336"], r"після двох тижнів", "Після двох тижнів навчального процесу на початку навчального року."),
    ("Який пароль від гостьової мережі Wi-Fi університету?", [], None, None),
]
extended_benchmark = BENCHMARK + extra_benchmark
extra_answerable = [(q, gold, evidence) for q, gold, evidence, _ in extra_benchmark if gold]
for question, gold, evidence in extra_answerable:
    matches = [doc for doc in chunks if doc.metadata["doc_id"] in gold and re.search(evidence, doc.page_content)]
    print(f"{question}: еталонних фрагментів {len(matches)}")
    if not matches:
        print("Увага: перевірити формулювання еталонного доказу.")
    else:
        print(matches[0].page_content)

def extended_metrics(search_method):
    ranks = [first_hit_rank(search_method(q, 10), gold, evidence) for q, gold, evidence in answerable + extra_answerable]
    return {"метод": search_method.__name__, "hit@1": hit_rate(ranks, 1), "hit@5": hit_rate(ranks, 5), "MRR": mrr(ranks)}

if all(any(doc.metadata["doc_id"] in gold and re.search(evidence, doc.page_content) for doc in chunks) for _, gold, evidence in extra_answerable):
    extended_table = pd.DataFrame([extended_metrics(method) for method in [vector_search, bm25_search, hybrid_search, rerank_search]])
    metrics_comparison = base_retrieval_metrics.merge(extended_table, on="метод", suffixes=("_base", "_extended"))
    print(f"Питань: {len(BENCHMARK)} → {len(extended_benchmark)}; з відповіддю: {len(answerable)} → {len(answerable) + len(extra_answerable)}")
    display(metrics_comparison.round(4))
else:
    print("Спочатку виправити еталонний доказ, потім повторити клітинку.")

Яка мінімальна частка вибіркових дисциплін для спеціальностей з додатковим регулюванням?: еталонних фрагментів 1
Положення про порядок та умови обрання здобувачами вищої освіти навчальних дисциплін. 1.2. Відповідно до Закону України «Про вищу освіту» (ст.62, п.15) та Положення про організацію освітнього процесу у Державному університеті «Житомирська політехніка» кожному здобувачу вищої освіти забезпечується право на вибір навчальних дисциплін у межах, передбачених відповідною освітньою програмою та навчальним планом, в обсязі, що становить не менш як 25 відсотків загальної кількості кредитів ЄКТС (не менше ніж 10 відсотків для спеціальностей, що передбачають доступ до професій, для яких запроваджено додаткове регулювання), передбачених для даного рівня вищої освіти. При цьому здобувачі певного рівня вищої освіти мають право вибирати навчальні дисципліни, що пропонуються для інших рівнів вищої освіти, за погодженням з керівником відповідного факультету чи структурного підрозділу. 1.3.
Н

,метод,hit@1_base,hit@5_base,MRR_base,hit@1_extended,hit@5_extended,MRR_extended
0,vector_search,0.6087,0.8696,0.7379,0.60,0.88,0.7269
1,bm25_search,0.5217,0.6522,0.5912,0.48,0.64,0.5705
2,hybrid_search,0.6087,0.9130,0.7569,0.60,0.92,0.7464
3,rerank_search,0.8261,0.9565,0.8967,0.84,0.96,0.9050


Після розширення набір містить 29 питань замість 26; питань з відповіддю — 25 замість 23. Обидва нові еталонні докази знайдені в індексі. Питання про пароль Wi-Fi позначено як питання без відповіді: пароль не передбачений тематикою обраних положень, але повна відсутність його у корпусі окремим пошуком не перевірялася.

| Метод | Hit@1 до / після | Hit@5 до / після | MRR@10 до / після |
|---|---|---|---|
| Semantic | 0,6087 / 0,6000 | 0,8696 / 0,8800 | 0,7379 / 0,7269 |
| BM25 | 0,5217 / 0,4800 | 0,6522 / 0,6400 | 0,5912 / 0,5705 |
| Hybrid | 0,6087 / 0,6000 | 0,9130 / 0,9200 | 0,7569 / 0,7464 |
| Rerank | 0,8261 / 0,8400 | 0,9565 / 0,9600 | 0,8967 / 0,9050 |

Rerank покращив усі три метрики, BM25 погіршив усі три, а Semantic і Hybrid трохи покращили Hit@5, але знизили Hit@1 та MRR. Питання без відповіді виключено з розрахунку ретриву.

### 28. Порівняйте метрики ретриву для чотирьох методів пошуку (розділ 7). Чому переранжування найсильніше впливає саме на hit@1?

BM25 працює за лексичними збігами, семантичний пошук — за близькістю ембедингів, hybrid поєднує обидва рейтинги через RRF. Rerank додатково оцінює пари запит–фрагмент і може підняти правильний фрагмент із нижчої позиції на першу. Тому Hit@1 може зрости сильніше за Hit@5: фрагмент уже був у п’ятірці, але став першим. Якщо правильний фрагмент не потрапив до кандидатів, reranker його не додасть.

In [25]:
display(base_retrieval_metrics.round(4))

,метод,hit@1,hit@5,MRR
0,vector_search,0.6087,0.8696,0.7379
1,bm25_search,0.5217,0.6522,0.5912
2,hybrid_search,0.6087,0.9130,0.7569
3,rerank_search,0.8261,0.9565,0.8967


| Метод | Hit@1 | Hit@5 | MRR@10 |
|---|---:|---:|---:|
| Semantic | 0,6087 | 0,8696 | 0,7379 |
| BM25 | 0,5217 | 0,6522 | 0,5912 |
| Hybrid | 0,6087 | 0,9130 | 0,7569 |
| Rerank | 0,8261 | 0,9565 | 0,8967 |

Найкращий результат у Rerank. Порівняно з Hybrid, Hit@1 зріс з 60,87% до 82,61% (+21,74 в. п.), Hit@5 — з 91,30% до 95,65% (+4,35 в. п.). Це узгоджується з тим, що переранжування переважно покращує позиції вже знайдених фрагментів. MRR обчислено для перших 10 результатів.

### 29. У розділі 9 суддею виступає та сама модель, що генерує відповіді. Які проблеми це може спричинити? Перевірте вручну вердикти судді для пʼяти відповідей і порівняйте зі своїми.

Та сама модель може повторювати власні помилки або віддавати перевагу знайомому стилю відповіді. Вона може пропустити неправильну умову чи відхилити правильне перефразування. Суддя тут порівнює відповідь з еталоном, а не перевіряє факти за першоджерелом.

In [26]:
reference_by_question = {q: (gold, evidence, reference) for q, gold, evidence, reference in BENCHMARK}
judged = evaluation[evaluation["документи"].map(bool)]
selected = pd.concat([judged[~judged["правильно"]].head(2), judged[judged["правильно"]]]).head(5)
if len(selected) < 5:
    selected = pd.concat([selected, judged.drop(index=selected.index)]).head(5)
for n, (_, row) in enumerate(selected.iterrows(), 1):
    question = row["питання"]
    print(f"{n}. {question}")
    print("Еталон:", reference_by_question[question][2])
    print("Відповідь:", row["відповідь"])
    print("Суддя:", "так" if row["правильно"] else "ні", "\n")

1. Для кого встановлюється стипендія ректора?
Еталон: Для студентів денної форми навчання із суттєвими досягненнями в навчанні, науковій, творчій і громадській діяльності.
Відповідь: Для студентів денної форми навчання, які мають суттєві досягнення у навчанні, науковій, творчій та громадській діяльності, є переможцями міжнародних та всеукраїнських олімпіад, конкурсів та змагань [1].
Суддя: ні 

2. Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?
Еталон: F2.
Відповідь: Шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі — 121 [2].
Суддя: ні 

3. До якої дати потрібно оплатити гуртожиток за перший семестр?
Еталон: До 1 вересня поточного року.
Відповідь: До 1 вересня поточного року потрібно оплатити гуртожиток за перший семестр [1].
Суддя: так 

4. До якої дати оплачують гуртожиток за другий семестр?
Еталон: До 1 лютого поточного року.
Відповідь: До 1 лютого поточного року [1].
Суддя: так 

5. З якої до якої години в г

Перевірка за еталонами та збереженими відповідями (для першої відповіді також є повний фрагмент у розділі аналізу відмов):

| Питання | Суддя | Оцінка після перевірки | Пояснення |
|---|---|---|---|
| Стипендія ректора | Ні | Так | Відповідь містить еталонні умови та додаткову інформацію про переможців з джерела [1]. |
| Шифр спеціальності | Ні | Ні | 121 замість F2. |
| Оплата гуртожитку за перший семестр | Так | Так | До 1 вересня, збігається з еталоном. |
| Оплата гуртожитку за другий семестр | Так | Так | До 1 лютого, збігається з еталоном. |
| Час тиші в гуртожитку | Так | Так | З 23:00 до 7:00, збігається з еталоном. |

Збіг вердиктів — 4 з 5. Суддя помилково відхилив відповідь про стипендію, хоча вона підтримується документом. 

### 30. Розгляньте неправильні відповіді в розділі 10 «Аналіз відмов». Для кожної визначте тип помилки: промах ретриву, помилка генерації, помилка судді чи неоднозначний еталон. Обґрунтуйте.

Для кожної помилки перевіряю еталон, відповідь і фактичні фрагменти, передані моделі. Наявність правильного ID документа ще не означає, що знайдено потрібний доказ. Якщо доказ не знайдений — можливий промах ретриву. Якщо доказ є, але відповідь неправильна — помилка генерації. Якщо відповідь правильна, а суддя її відхилив — помилка судді. Якщо є інше обґрунтоване джерело чи суперечність у документах — перевіряю неоднозначність еталона. Для питань без відповіді перевіряю коректність відмови.

In [27]:
print(f"Кандидатів на аналіз: {len(failures)}")
for n, (_, row) in enumerate(failures.iterrows(), 1):
    question = row["питання"]
    gold, evidence, reference = reference_by_question[question]
    retrieved_docs = evaluation_contexts[question]
    rank = first_hit_rank(retrieved_docs, gold, evidence) if gold else None
    print(f"\n{n}. {question}")
    print("Еталон:", reference if gold else "Відповіді немає")
    print("Відповідь:", row["відповідь"])
    print("Вердикт:", row["правильно"], "Цитування:", row["цитування"], "Позиція доказу:", rank)
    for i, doc in enumerate(retrieved_docs, 1):
        print(f"[{i}] {doc.metadata['doc_id']} | {doc.page_content}")

Кандидатів на аналіз: 2

1. Для кого встановлюється стипендія ректора?
Еталон: Для студентів денної форми навчання із суттєвими досягненнями в навчанні, науковій, творчій і громадській діяльності.
Відповідь: Для студентів денної форми навчання, які мають суттєві досягнення у навчанні, науковій, творчій та громадській діяльності, є переможцями міжнародних та всеукраїнських олімпіад, конкурсів та змагань [1].
Вердикт: False Цитування: True Позиція доказу: 1
[1] 344 | Положення про стипендію ректора. ЗМІСТ 1. Загальні положення……………………………….. 3 2. Порядок призначення стипендії………………… 3 ЗМІСТ 1. Загальні положення……………………………….. 3 2. Порядок призначення стипендії………………… 3 1. Загальні положення 1.1. Стипендія Ректора Державного університету «Житомирська політехніка» (у подальшому - Стипендія) встановлюється для студентів денної форми навчання які мають суттєві досягнення у навчанні, науковій, творчій та громадській діяльності, є переможцями міжнародних та всеукраїнських олімпіад, конкурсів та

Автоматичне оцінювання: 21/23 правильних відповідей, 22/23 збігів цитованого ID документа з еталоном, 3/3 правильні відмови. Вивід містить два проблемні випадки:

1. **Стипендія ректора — помилка судді.** Еталонний доказ на першій позиції. Відповідь містить умови з положення та додаткову підтверджену інформацію про переможців олімпіад, конкурсів і змагань. Суддя відхилив її, хоча вона відповідає джерелу.
2. **Шифр спеціальності — промах ретриву на рівні доказу та суперечливі джерела.** Еталон F2 не знайдений за заданим критерієм у Top-5. Чотири фрагменти належать старій програмі 2024 року, а знайдений фрагмент чинної програми 2025 року також містить старий шифр 121. Модель відповіла 121 і послалася на старе джерело [2]. Потрібні фільтрація за актуальністю й виправлення суперечності в чинному документі.

Після перегляду двох випадків змістова оцінка відповідей становить 22/23 (95,65%), якщо решту вердиктів залишити без змін.

### Загальний час виконання

In [28]:
_demo_total_seconds = _demo_clock() - _demo_start
print(f"Основне демо: {_demo_main_seconds:.2f} с")
print(f"Додаткові завдання: {_demo_total_seconds - _demo_main_seconds:.2f} с")
print(f"Загалом: {_demo_total_seconds:.2f} с ({_demo_total_seconds / 60:.2f} хв)")

Основне демо: 371.59 с
Додаткові завдання: 10.38 с
Загалом: 381.97 с (6.37 хв)
